# Estabilidade e escolha do modelo

Uma partição só serve para descrever perfis se não mudar muito quando os
dados mudam um pouco. Este notebook executa os três testes de estabilidade
previstos no método (subjanelas de dois anos, retirada dos municípios
pequenos e, para o K-means, sementes diferentes), monta a tabela de
comparação dos algoritmos e aplica a regra de decisão do grupo.

Saídas:

- `data/processed/tabela_comparacao_algoritmos.csv`: as 18 partições do
  notebook 04 com as medidas de qualidade e de estabilidade, mais uma linha
  de resumo do DBSCAN;
- `data/processed/perfis.csv`: a partição escolhida, gerada **somente depois**
  que o grupo preencher `ALGORITMO_ESCOLHIDO` e `K_ESCOLHIDO` no `config.py`.

### Regra de decisão

Regra definida antes da execução dos testes. Versão de 02/10: os passos 1 a 5
são os do plano; o passo 4b e a tolerância de 0,05 foram acrescentados em
02/10 e confirmados pelo grupo antes da execução.

1. **Candidatos**: K-means e Ward com k de 2 a 10. O DBSCAN só entra se
   alguma configuração da grade tiver pelo menos 2 grupos e menos de 20% de
   ruído.
2. **Filtro de tamanho**: sai a partição com algum grupo menor que 5% dos
   municípios (32 municípios).
3. **Qualidade**: entre as que sobraram, para cada algoritmo ficam os k que
   estão entre os 3 melhores em silhueta **e** entre os 3 melhores em
   Calinski–Harabasz. Se nenhum k satisfizer as duas, ficam os 3 melhores em
   silhueta. O Davies–Bouldin é reportado e serve de desempate.
4. **Estabilidade**: os candidatos são ordenados pelo `ari_estabilidade`
   (média dos três ARI: as duas subjanelas e a retirada dos municípios
   pequenos). Diferenças menores que **0,05** contam como empate.
   - **4b. Dois níveis.** A melhor partição com k = 2 é registrada como a
     **divisão mais robusta** e entra na resposta da pergunta (ii). Uma
     divisão em dois grupos, porém, não forma "perfis" no sentido da pergunta
     (iii). Por isso, a partição usada para caracterizar os perfis é a mais
     estável **com k ≥ 3**, pelos mesmos critérios.
5. **Interpretabilidade**: entre empatados, o grupo escolhe a partição cujos
   perfis todos conseguem descrever em uma frase, e registra a justificativa.

O código aplica os passos 1 a 4b e apenas **sugere**; o passo 5 cabe ao
grupo.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Os notebooks ficam em notebooks/ e o código do projeto em src/. Estas duas
# linhas apontam o Python para src/, para os "from config import ..." abaixo
# funcionarem tanto rodando daqui quanto da raiz do projeto.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from config import (ALGORITMO_ESCOLHIDO, BASE_FINAL_CSV, DATA_PROCESSED,
                    DICIONARIO_CSV, K_ESCOLHIDO, K_MIN_PERFIS,
                    N_MELHORES_QUALIDADE, RUIDO_MAXIMO_DBSCAN,
                    SSP_COBERTURA_CSV, SSP_PAINEL_CSV, SUBJANELAS,
                    TAMANHO_MINIMO_GRUPO, TOLERANCIA_ARI)
from estilo import aplicar_estilo
from preprocessamento import (carregar_features, exposicao_da_janela,
                              montar_matriz)
from clusterizacao import (agrupar, ari_sem_pequenos, ari_sementes,
                           ari_subjanela, distancia_ao_centro,
                           marcar_qualidade, matriz_subjanela,
                           renumerar_por_socioeconomico)

aplicar_estilo()   # mesma fonte, cores e grade em todas as figuras do artigo
# Se algum import acima falhar, quase sempre é o editor apontando para outra
# instalação do Python. O caminho impresso aqui é o que precisa ter as
# bibliotecas do requirements.txt.
print("Python:", sys.executable)

# O código do IBGE é identificador, não quantidade: lido como texto para não
# virar número em nenhuma etapa.
base = pd.read_csv(BASE_FINAL_CSV, dtype={"codigo_ibge": str})
dic = pd.read_csv(DICIONARIO_CSV)
painel = pd.read_csv(SSP_PAINEL_CSV, dtype={"codigo_ibge": str})
cobertura = pd.read_csv(SSP_COBERTURA_CSV)
matriz = pd.read_csv(DATA_PROCESSED / "matriz_modelagem.csv",
                     dtype={"codigo_ibge": str})
# Saídas do notebook 04.
metricas = pd.read_csv(DATA_PROCESSED / "metricas_clusterizacao.csv")
grade = pd.read_csv(DATA_PROCESSED / "dbscan_grade.csv")

Python: C:\Users\Higor\Documents\6 Sem\RP2\projeto\database-rp2\.venv\Scripts\python.exe


## 1. Por que testar a estabilidade

Os testes refazem o agrupamento com dados um pouco diferentes e comparam o
resultado com a partição original. A comparação usa o **ARI** (índice de Rand
ajustado), que verifica, para cada par de municípios, se os dois continuam
juntos ou separados nas duas partições. O ARI vale 1 quando as partições são
iguais e fica perto de 0 quando concordam tanto quanto um sorteio. O número
que cada grupo recebe não importa: o ARI considera apenas quem está junto de
quem.

As partições de referência são as do notebook 04, refeitas aqui sobre a
mesma matriz. A conferência abaixo garante que a matriz montada pela função
de `src/preprocessamento.py` é idêntica ao `matriz_modelagem.csv`.

In [2]:
features, bloco_de = carregar_features(dic)
# reset_index para as posições baterem com as dos rótulos do scikit-learn.
modelagem = base[~base["flag_sem_iegm"]].reset_index(drop=True)
Z, W, peso = montar_matriz(modelagem, features, bloco_de)
assert np.allclose(W, matriz[features]), "W diferente do matriz_modelagem.csv"

# Uma partição de referência para cada algoritmo e cada k do notebook 04.
particoes = list(metricas[["algoritmo", "k"]].itertuples(index=False))
referencia = {(alg, k): agrupar(W, alg, k) for alg, k in particoes}
print(f"{len(referencia)} partições de referência sobre {len(W)} municípios")

18 partições de referência sobre 644 municípios


## 2. Subjanelas

A janela completa cobre 2023 a 2025. As subjanelas 2023–2024 e 2024–2025
recalculam as taxas criminais com apenas dois anos cada; as variáveis
socioeconômicas e de gestão não mudam. As taxas continuam por 100 mil
habitantes **por ano**, pois a soma das ocorrências é dividida pela
exposição, que passa a ser de 2 anos. Se a partição se mantiver, ela não
depende de um ano específico.

A matriz de cada subjanela é refeita do início pela mesma função do notebook
03 (`montar_matriz`) e montada uma única vez, servindo para todas as 18
partições.

In [3]:
matrizes_sub = {}
for anos in SUBJANELAS:
    # Todos os anos têm 12 meses, então cada subjanela vale 2 anos de exposição.
    assert exposicao_da_janela(cobertura, anos) == 2.0
    matrizes_sub[f"ari_{anos[0]}_{anos[1]}"] = matriz_subjanela(
        modelagem, painel, cobertura, features, bloco_de, anos)

# Uma linha por partição, uma coluna por subjanela.
ari_sub = pd.DataFrame([
    {"algoritmo": alg, "k": k,
     **{nome: ari_subjanela(W_sub, alg, k, referencia[(alg, k)])
        for nome, W_sub in matrizes_sub.items()}}
    for alg, k in particoes])
ari_sub.round(3)

,algoritmo,k,ari_2023_2024,ari_2024_2025
0,kmeans,2,0.897,0.909
1,kmeans,3,0.818,0.901
2,kmeans,4,0.891,0.912
3,kmeans,5,0.850,0.835
4,kmeans,6,0.836,0.792
5,kmeans,7,0.769,0.687
6,kmeans,8,0.753,0.512
7,kmeans,9,0.599,0.697
8,kmeans,10,0.455,0.407
9,ward,2,0.432,0.509


## 3. Sem os municípios pequenos

Em um município com menos de 5.000 habitantes, uma única ocorrência a mais ou
a menos muda bastante a taxa por 100 mil. O teste retira esses municípios
(`flag_pop_pequena`), refaz a padronização e o agrupamento só com os que
sobram, como se os pequenos nunca tivessem existido, e compara o resultado com
a partição original restrita aos mesmos municípios.

In [4]:
n_ficam = int((~modelagem["flag_pop_pequena"]).sum())
print(f"ficam {n_ficam} municípios")

ari_peq = pd.DataFrame([
    {"algoritmo": alg, "k": k,
     "ari_sem_pequenos": ari_sem_pequenos(modelagem, features, bloco_de,
                                          alg, k, referencia[(alg, k)])}
    for alg, k in particoes])
ari_peq.round(3)

ficam 495 municípios


,algoritmo,k,ari_sem_pequenos
0,kmeans,2,0.744
1,kmeans,3,0.512
2,kmeans,4,0.480
3,kmeans,5,0.578
4,kmeans,6,0.782
5,kmeans,7,0.560
6,kmeans,8,0.532
7,kmeans,9,0.468
8,kmeans,10,0.479
9,ward,2,0.441


## 4. Sementes (só K-means)

O K-means começa sorteando os centros, e a partição de referência é a melhor
de 50 sorteios. O teste roda o K-means outras 20 vezes, cada uma com um único
sorteio e uma semente diferente, e compara cada resultado com a referência.
Um ARI médio alto indica que o K-means chega quase sempre à mesma partição;
um ARI baixo indica que o resultado depende do sorteio. O Ward não sorteia
nada e não passa por este teste.

Este teste não entra no `ari_estabilidade` da regra de decisão; é reportado
como informação adicional.

In [5]:
# ari_sementes devolve dois números (média e mínimo); o zip(*...) separa
# as duas listas.
ks = metricas.loc[metricas["algoritmo"] == "kmeans", "k"].tolist()
medias, minimos = zip(*[ari_sementes(W, k) for k in ks])
ari_sem = pd.DataFrame({"algoritmo": "kmeans", "k": ks,
                        "ari_sementes_media": medias,
                        "ari_sementes_min": minimos})
ari_sem.round(3)

,algoritmo,k,ari_sementes_media,ari_sementes_min
0,kmeans,2,0.981,0.969
1,kmeans,3,0.451,0.269
2,kmeans,4,0.929,0.850
3,kmeans,5,0.679,0.371
4,kmeans,6,0.584,0.360
5,kmeans,7,0.560,0.350
6,kmeans,8,0.533,0.388
7,kmeans,9,0.490,0.318
8,kmeans,10,0.439,0.316


Com um único sorteio, o K-means reencontra praticamente a mesma partição
em k = 2 (ARI médio de 0,981) e em k = 4 (0,929). Nos demais k, o ARI médio
fica entre 0,439 e 0,679; em k = 3 é de 0,451, com mínimo de 0,269. O
resultado não altera a partição de referência, que é a melhor de 50 sorteios
com semente fixa, mas indica que, em k = 3, o K-means pode chegar a soluções
diferentes a depender do ponto de partida.

## 5. Tabela de comparação

A tabela reúne as medidas de qualidade do notebook 04 e os testes de
estabilidade. `ari_estabilidade` é a média dos três ARI usados na regra.
`passa_tamanho` e `candidato_qualidade` registram os passos 2 e 3 da regra,
aplicados na seção 6. A última linha resume o DBSCAN.

In [6]:
tabela = (metricas.drop(columns=["soma_quadrados", "maior_grupo"])
          .merge(ari_sub, on=["algoritmo", "k"])
          .merge(ari_peq, on=["algoritmo", "k"]))
tabela["ari_estabilidade"] = tabela[
    ["ari_2023_2024", "ari_2024_2025", "ari_sem_pequenos"]].mean(axis=1)
# how="left": o Ward fica com NaN nas colunas de sementes.
tabela = tabela.merge(ari_sem, on=["algoritmo", "k"], how="left")

# Passo 2: um grupo com menos de 5% dos 644 municípios (32,2) reprova a partição.
tabela["passa_tamanho"] = tabela["menor_grupo"] >= TAMANHO_MINIMO_GRUPO * len(W)
# Passo 3: só entre as que passaram no tamanho.
tabela["candidato_qualidade"] = False
passaram = tabela[tabela["passa_tamanho"]]
tabela.loc[passaram.index, "candidato_qualidade"] = marcar_qualidade(
    passaram, N_MELHORES_QUALIDADE)

In [7]:
# Passo 1 da regra: o DBSCAN só entra com 2 grupos ou mais e menos de 20% de
# ruído. A linha de resumo mostra a configuração com mais grupos (e, entre
# essas, a de menos ruído).
admitidas = grade[(grade["n_grupos"] >= 2)
                  & (grade["pct_ruido"] < 100 * RUIDO_MAXIMO_DBSCAN)]
melhor = grade.sort_values(["n_grupos", "pct_ruido"],
                           ascending=[False, True]).iloc[0]
linha_dbscan = {
    "algoritmo": "dbscan", "k": int(melhor["n_grupos"]),
    "silhueta": melhor["silhueta"], "passa_tamanho": False,
    "candidato_qualidade": False,
    "observacao": (f"melhor da grade: min_pts={int(melhor['min_pts'])}, "
                   f"eps={melhor['eps']:.3f}, ruído {melhor['pct_ruido']:.1f}%; "
                   f"admitido no passo 1: {'sim' if len(admitidas) else 'não'}"),
}
tabela_saida = pd.concat([tabela, pd.DataFrame([linha_dbscan])],
                         ignore_index=True)
# Int64 é o inteiro do pandas que aceita vazio (a linha do DBSCAN).
tabela_saida["menor_grupo"] = tabela_saida["menor_grupo"].astype("Int64")
tabela_saida.to_csv(DATA_PROCESSED / "tabela_comparacao_algoritmos.csv",
                    index=False)
tabela_saida.drop(columns="observacao").round(3)

,algoritmo,k,silhueta,calinski_harabasz,davies_bouldin,menor_grupo,ari_2023_2024,ari_2024_2025,ari_sem_pequenos,ari_estabilidade,ari_sementes_media,ari_sementes_min,passa_tamanho,candidato_qualidade
0,kmeans,2,0.136,119.029,2.216,302,0.897,0.909,0.744,0.850,0.981,0.969,True,True
1,kmeans,3,0.129,92.110,2.210,76,0.818,0.901,0.512,0.743,0.451,0.269,True,True
2,kmeans,4,0.101,83.203,2.169,66,0.891,0.912,0.480,0.761,0.929,0.850,True,True
3,kmeans,5,0.090,72.684,2.373,67,0.850,0.835,0.578,0.755,0.679,0.371,True,False
4,kmeans,6,0.088,64.487,2.394,64,0.836,0.792,0.782,0.803,0.584,0.360,True,False
5,kmeans,7,0.089,58.166,2.326,21,0.769,0.687,0.560,0.672,0.560,0.350,False,False
6,kmeans,8,0.084,53.311,2.253,23,0.753,0.512,0.532,0.599,0.533,0.388,False,False
7,kmeans,9,0.077,49.643,2.311,21,0.599,0.697,0.468,0.588,0.490,0.318,False,False
8,kmeans,10,0.073,46.394,2.336,20,0.455,0.407,0.479,0.447,0.439,0.316,False,False
9,ward,2,0.114,100.546,2.297,238,0.432,0.509,0.441,0.461,NaN,NaN,True,True


O K-means é mais estável que o Ward em todos os k: o `ari_estabilidade` vai
de 0,447 a 0,850 no K-means e de 0,311 a 0,477 no Ward. Nas duas subjanelas,
o K-means mantém ARI acima de 0,75 de k = 2 a k = 6, e a retirada dos
municípios pequenos é o teste que mais altera as suas partições de k = 2 a
k = 7. No Ward, nenhum dos três ARI passa de 0,55.

## 6. Aplicação da regra

Cada célula aplica um passo e mostra as partições que restam.

In [8]:
# Passo 1: candidatos.
print(f"DBSCAN: {len(admitidas)} configuração(ões) com 2 grupos ou mais e "
      f"menos de {RUIDO_MAXIMO_DBSCAN:.0%} de ruído")
if len(admitidas):
    print("ATENÇÃO: o DBSCAN foi admitido e precisa entrar na tabela à mão.")
candidatos = tabela[tabela["algoritmo"].isin(["kmeans", "ward"])]
print(f"{len(candidatos)} candidatos: K-means e Ward com k de 2 a 10")

DBSCAN: 0 configuração(ões) com 2 grupos ou mais e menos de 20% de ruído
18 candidatos: K-means e Ward com k de 2 a 10


In [9]:
# Passo 2: filtro de tamanho.
candidatos = candidatos[candidatos["passa_tamanho"]]
candidatos[["algoritmo", "k", "menor_grupo"]]

,algoritmo,k,menor_grupo
0,kmeans,2,302
1,kmeans,3,76
2,kmeans,4,66
3,kmeans,5,67
4,kmeans,6,64
9,ward,2,238
10,ward,3,73
11,ward,4,73
12,ward,5,73
13,ward,6,73


In [10]:
# Passo 3: qualidade. O Davies-Bouldin aparece para servir de desempate.
candidatos = candidatos[candidatos["candidato_qualidade"]]
candidatos[["algoritmo", "k", "silhueta", "calinski_harabasz",
            "davies_bouldin"]].round(3)

,algoritmo,k,silhueta,calinski_harabasz,davies_bouldin
0,kmeans,2,0.136,119.029,2.216
1,kmeans,3,0.129,92.110,2.210
2,kmeans,4,0.101,83.203,2.169
9,ward,2,0.114,100.546,2.297
10,ward,3,0.112,81.609,2.297
11,ward,4,0.094,69.191,2.439


In [11]:
# Passo 4: estabilidade, do mais estável para o menos estável.
candidatos = candidatos.sort_values("ari_estabilidade", ascending=False)
candidatos[["algoritmo", "k", "ari_2023_2024", "ari_2024_2025",
            "ari_sem_pequenos", "ari_estabilidade", "davies_bouldin",
            "ari_sementes_media"]].round(3)

,algoritmo,k,ari_2023_2024,ari_2024_2025,ari_sem_pequenos,ari_estabilidade,davies_bouldin,ari_sementes_media
0,kmeans,2,0.897,0.909,0.744,0.850,2.216,0.981
2,kmeans,4,0.891,0.912,0.480,0.761,2.169,0.929
1,kmeans,3,0.818,0.901,0.512,0.743,2.210,0.451
10,ward,3,0.443,0.549,0.440,0.477,2.297,NaN
9,ward,2,0.432,0.509,0.441,0.461,2.297,NaN
11,ward,4,0.324,0.346,0.315,0.329,2.439,NaN


In [12]:
# Passo 4b, primeiro nível: a melhor partição com k = 2.
k2 = candidatos[candidatos["k"] == 2]
divisao_robusta = k2.iloc[0]   # já ordenado pelo ari_estabilidade
print(f"Divisão mais robusta (k = 2): {divisao_robusta['algoritmo']}, "
      f"ari_estabilidade = {divisao_robusta['ari_estabilidade']:.3f}")
# Diferença para a melhor: abaixo da tolerância conta como empate.
k2 = k2.assign(diferenca=divisao_robusta["ari_estabilidade"] - k2["ari_estabilidade"])
k2[["algoritmo", "k", "ari_estabilidade", "diferenca"]].round(3)

Divisão mais robusta (k = 2): kmeans, ari_estabilidade = 0.850


,algoritmo,k,ari_estabilidade,diferenca
0,kmeans,2,0.850,0.000
9,ward,2,0.461,0.389


In [13]:
# Passo 4b, segundo nível: as partições com k >= 3, para os perfis.
perfis_cand = candidatos[candidatos["k"] >= K_MIN_PERFIS].copy()
melhor_ari = perfis_cand["ari_estabilidade"].max()
perfis_cand["diferenca"] = melhor_ari - perfis_cand["ari_estabilidade"]
perfis_cand["empate"] = perfis_cand["diferenca"] < TOLERANCIA_ARI

print("Sugestão para os perfis (k >= 3), empatadas pela tolerância de "
      f"{TOLERANCIA_ARI}:")
for _, c in perfis_cand[perfis_cand["empate"]].iterrows():
    print(f"  {c['algoritmo']} k = {c['k']}: ari_estabilidade = "
          f"{c['ari_estabilidade']:.3f} (diferença {c['diferenca']:.3f})")
perfis_cand[["algoritmo", "k", "ari_estabilidade", "diferenca", "empate",
             "davies_bouldin", "ari_sementes_media"]].round(3)

Sugestão para os perfis (k >= 3), empatadas pela tolerância de 0.05:
  kmeans k = 4: ari_estabilidade = 0.761 (diferença 0.000)
  kmeans k = 3: ari_estabilidade = 0.743 (diferença 0.018)


,algoritmo,k,ari_estabilidade,diferenca,empate,davies_bouldin,ari_sementes_media
2,kmeans,4,0.761,0.000,True,2.169,0.929
1,kmeans,3,0.743,0.018,True,2.210,0.451
10,ward,3,0.477,0.284,False,2.297,NaN
11,ward,4,0.329,0.433,False,2.439,NaN


- **Passo 1.** Nenhuma configuração do DBSCAN tem 2 grupos ou mais, e o
  algoritmo fica fora; restam 18 candidatos.
- **Passo 2.** Saem o K-means com k de 7 a 10 e o Ward com k de 8 a 10, por
  terem algum grupo com menos de 5% dos municípios; restam 11.
- **Passo 3.** Nos dois algoritmos, k = 2, 3 e 4 estão entre os 3 melhores
  tanto em silhueta quanto em Calinski–Harabasz; restam 6.
- **Passo 4b, divisão mais robusta.** K-means com k = 2
  (`ari_estabilidade` de 0,850, contra 0,461 do Ward com k = 2).
- **Passo 4b, perfis.** K-means com k = 4 (0,761) e K-means com k = 3
  (0,743) ficam empatados: a diferença, de 0,018, é menor que a tolerância de
  0,05. O Ward com k = 3 (0,477) e com k = 4 (0,329) fica fora.

A escolha entre k = 3 e k = 4 do K-means cabe ao grupo, pelo critério de
interpretabilidade (passo 5). Duas informações da tabela podem subsidiá-la:
o Davies–Bouldin, que a regra usa como desempate no passo 3, é ligeiramente
melhor em k = 4 (2,169 contra 2,210), e o teste de sementes mostrou que k = 4
é reencontrado com mais facilidade (ARI médio de 0,929 contra 0,451).

## 7. Partição escolhida

A escolha entre as partições empatadas (passo 5) é do grupo e fica registrada
em `ALGORITMO_ESCOLHIDO` e `K_ESCOLHIDO`, no `config.py`. Enquanto as duas
constantes estiverem vazias, a célula abaixo não salva nada.

Depois da escolha, a célula agrupa os municípios com o modelo escolhido e
renumera os perfis em ordem crescente da média das 6 variáveis
socioeconômicas padronizadas, de modo que o perfil 1 seja o de menor condição
socioeconômica. O `perfis.csv` traz também a distância de cada município ao
centro do seu perfil, usada no notebook 06 para encontrar os municípios mais
típicos, e a coluna `divisao_k2`, com a divisão mais robusta do passo 4b,
numerada pelo mesmo critério.

In [14]:
if ALGORITMO_ESCOLHIDO is None or K_ESCOLHIDO is None:
    print("escolha pendente: preencher no config.py depois da reunião")
else:
    colunas_socio = [c for c in features if bloco_de[c] == "socioeconomico"]
    # A numeração do scikit-learn é arbitrária; renumeramos pela condição
    # socioeconômica para "perfil 1" querer dizer sempre a mesma coisa.
    perfil = renumerar_por_socioeconomico(
        agrupar(W, ALGORITMO_ESCOLHIDO, K_ESCOLHIDO), Z, colunas_socio)
    divisao_k2 = renumerar_por_socioeconomico(
        agrupar(W, divisao_robusta["algoritmo"], 2), Z, colunas_socio)
    perfis = pd.DataFrame({
        "codigo_ibge": modelagem["codigo_ibge"],
        "municipio": modelagem["municipio"],
        "perfil": perfil,
        "distancia_ao_centro": distancia_ao_centro(W, perfil),
        "divisao_k2": divisao_k2,
    })
    perfis.to_csv(DATA_PROCESSED / "perfis.csv", index=False)
    print(f"{ALGORITMO_ESCOLHIDO}, k = {K_ESCOLHIDO} -> perfis.csv")
    print(perfis["perfil"].value_counts().sort_index())

escolha pendente: preencher no config.py depois da reunião
